# Tree Ensembles

Using grid-tuned XGBoost and Random Forest models for classification of `event_adverse` target.

In [1]:
# Import libraries

import pandas as pd
import os
from dotenv import load_dotenv
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV

)

In [2]:
# Load environment variables for GPU XGBoost training

load_dotenv()

device = os.getenv("GPU") or "cpu"

In [3]:
# Load the processed dataset

df = pd.read_csv("cleaned_data_final.csv")

In [4]:
# Separate and train test split data from target

X = df.drop(columns=["event_adverse"])
y = df["event_adverse"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [5]:
# Specify scoring metrics

from sklearn.metrics import make_scorer
from sklearn.metrics import precision_score, recall_score, f1_score

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(
        precision_score,
        zero_division=0
    ),
    "recall": make_scorer(
        recall_score,
        zero_division=0
    ),
    "f1": make_scorer(
        f1_score,
        zero_division=0
    ),
    "roc_auc": "roc_auc",
}

In [6]:
# XGBoost model setup

scale_pos_weight = (
    (y_train == 0).sum() / (y_train == 1).sum()
)

xgb_model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    device=device,

)

xgboost_param_grid = {
    "n_estimators": [30, 150, 300],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.01, 0.05, 0.1],
}

xgboost_grid = GridSearchCV(
    xgb_model,
    xgboost_param_grid,
    scoring=scoring,
    refit="f1",
    cv=5,
    n_jobs=1,
    verbose=3
)

In [7]:
# RF Model setup

rf_model = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

rf_param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [5, 10],
    "min_samples_split": [2, 5],
}

rf_grid = GridSearchCV(
    rf_model,
    rf_param_grid,
    scoring=scoring,
    refit="f1",       # or "recall"
    cv=5,
    n_jobs=1,
    verbose=3
)

In [8]:
# Train the XGBoost model

xgboost_grid.fit(X_train, y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


/home/naya/healthcare-1e/.venv/lib/python3.14/site-packages/xgboost/core.py:569: UserWarning: [15:53:36] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[CV 1/5] END learning_rate=0.01, max_depth=3, n_estimators=30; accuracy: (test=0.596) f1: (test=0.343) precision: (test=0.222) recall: (test=0.747) roc_auc: (test=0.711) total time=   0.5s
[CV 2/5] END learning_rate=0.01, max_depth=3, n_estimators=30; accuracy: (test=0.603) f1: (test=0.346) precision: (test=0.225) recall: (test=0.747) roc_auc: (test=0.714) total time=   0.3s
[CV 3/5] END learning_rate=0.01, max_depth=3, n_estimators=30; accuracy: (test=0.602) f1: (test=0.343) precision: (test=0.224) recall: (test=0.737) roc_auc: (test=0.706) total time=   0.2s
[CV 4/5] END learning_rate=0.01, max_depth=3, n_estimators=30; accuracy: (test=0.601) f1: (test=0.342) precision: (test=0.223) recall: (test=0.736) roc_auc: (test=0.707) total time=   0.2s
[CV 5/5] END learning_rate=0.01, max_depth=3, n_estimators=30; accuracy: (test=0.596) f1: (test=0.345) precision: (test=0.223) recall: (test=0.755) roc_auc: (test=0.711) total time=   0.2s
[CV 1/5] END learning_rate=0.01, max_depth=3, n_estimat

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 4, ...], 'n_estimators': [30, 150, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'f1': make_scorer(f...ro_division=0), 'precision': make_scorer(p...ro_division=0), 'recall': make_scorer(r...ro_division=0), ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strat

In [9]:
# Train the RF model

rf_grid.fit(X_train, y_train)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV 1/5] END max_depth=5, min_samples_split=2, n_estimators=100; accuracy: (test=0.623) f1: (test=0.351) precision: (test=0.232) recall: (test=0.725) roc_auc: (test=0.720) total time=   1.4s
[CV 2/5] END max_depth=5, min_samples_split=2, n_estimators=100; accuracy: (test=0.625) f1: (test=0.355) precision: (test=0.234) recall: (test=0.732) roc_auc: (test=0.726) total time=   1.4s
[CV 3/5] END max_depth=5, min_samples_split=2, n_estimators=100; accuracy: (test=0.616) f1: (test=0.348) precision: (test=0.229) recall: (test=0.726) roc_auc: (test=0.715) total time=   1.4s
[CV 4/5] END max_depth=5, min_samples_split=2, n_estimators=100; accuracy: (test=0.624) f1: (test=0.350) precision: (test=0.231) recall: (test=0.719) roc_auc: (test=0.718) total time=   1.4s
[CV 5/5] END max_depth=5, min_samples_split=2, n_estimators=100; accuracy: (test=0.623) f1: (test=0.354) precision: (test=0.233) recall: (test=0.732) roc_auc: (test=0.722) tota

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [5, 10], 'min_samples_split': [2, 5], 'n_estimators': [100, 300]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'f1': make_scorer(f...ro_division=0), 'precision': make_scorer(p...ro_division=0), 'recall': make_scorer(r...ro_division=0), ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be

In [10]:
# Show XGBoost results

xgb_results = pd.DataFrame(xgboost_grid.cv_results_)

xgb_table = xgb_results[
    [
        "param_n_estimators",
        "param_max_depth",
        "param_learning_rate",
        "mean_test_accuracy",
        "std_test_accuracy",
        "mean_test_precision",
        "std_test_precision",
        "mean_test_recall",
        "std_test_recall",
        "mean_test_f1",
        "std_test_f1",
        "mean_test_roc_auc",
        "std_test_roc_auc",
    ]
].copy()

xgb_table = xgb_table.rename(columns={
    "param_n_estimators": "n_estimators",
    "param_max_depth": "max_depth",
    "param_learning_rate": "learning_rate",

    "mean_test_accuracy": "accuracy",
    "std_test_accuracy": "accuracy_std",

    "mean_test_precision": "precision",
    "std_test_precision": "precision_std",

    "mean_test_recall": "recall",
    "std_test_recall": "recall_std",

    "mean_test_f1": "f1",
    "std_test_f1": "f1_std",

    "mean_test_roc_auc": "roc_auc",
    "std_test_roc_auc": "roc_auc_std",
})

xgb_table = xgb_table.sort_values(
    "f1",
    ascending=False
).reset_index(drop=True)

display(
    xgb_table.style.format({
        "learning_rate": "{:.3f}",
        "accuracy": "{:.6f}",
        "accuracy_std": "{:.4f}",
        "precision": "{:.6f}",
        "precision_std": "{:.4f}",
        "recall": "{:.6f}",
        "recall_std": "{:.4f}",
        "f1": "{:.6f}",
        "f1_std": "{:.4f}",
        "roc_auc": "{:.6f}",
        "roc_auc_std": "{:.4f}",
    })
)

,n_estimators,max_depth,learning_rate,accuracy,accuracy_std,precision,precision_std,recall,recall_std,f1,f1_std,roc_auc,roc_auc_std
0,300,5,0.100,0.662287,0.0012,0.252274,0.0015,0.710955,0.0056,0.372403,0.0024,0.744016,0.0031
1,150,5,0.100,0.659324,0.0016,0.251306,0.0016,0.716081,0.0049,0.372044,0.0024,0.743814,0.0030
2,300,4,0.100,0.660059,0.0017,0.251436,0.0014,0.714181,0.0046,0.371928,0.0020,0.743896,0.0029
3,300,5,0.050,0.659298,0.0016,0.251228,0.0016,0.715722,0.0043,0.371910,0.0023,0.744165,0.0030
4,300,4,0.050,0.657947,0.0016,0.250706,0.0017,0.717568,0.0058,0.371585,0.0025,0.742806,0.0029
5,150,4,0.100,0.658179,0.0018,0.250709,0.0016,0.716744,0.0048,0.371477,0.0022,0.742738,0.0029
6,300,3,0.100,0.657154,0.0014,0.250131,0.0015,0.717102,0.0061,0.370890,0.0023,0.742490,0.0031
7,150,5,0.050,0.655252,0.0021,0.249339,0.0018,0.719253,0.0046,0.370305,0.0026,0.742100,0.0029
8,300,3,0.050,0.653479,0.0018,0.248248,0.0017,0.719217,0.0052,0.369095,0.0024,0.740223,0.0028
9,150,3,0.100,0.653484,0.0014,0.248118,0.0016,0.718464,0.0057,0.368853,0.0025,0.740124,0.0030


In [11]:
# Show RF results

rf_results = pd.DataFrame(rf_grid.cv_results_)

rf_table = rf_results[
    [
        "param_n_estimators",
        "param_max_depth",
        "param_min_samples_split",
        "mean_test_accuracy",
        "std_test_accuracy",
        "mean_test_precision",
        "std_test_precision",
        "mean_test_recall",
        "std_test_recall",
        "mean_test_f1",
        "std_test_f1",
        "mean_test_roc_auc",
        "std_test_roc_auc",
    ]
].copy()

rf_table = rf_table.rename(columns={
    "param_n_estimators": "n_estimators",
    "param_max_depth": "max_depth",
    "param_min_samples_split": "min_samples_split",

    "mean_test_accuracy": "accuracy",
    "std_test_accuracy": "accuracy_std",

    "mean_test_precision": "precision",
    "std_test_precision": "precision_std",

    "mean_test_recall": "recall",
    "std_test_recall": "recall_std",

    "mean_test_f1": "f1",
    "std_test_f1": "f1_std",

    "mean_test_roc_auc": "roc_auc",
    "std_test_roc_auc": "roc_auc_std",
})

# Sort by F1 because refit="f1"
rf_table = rf_table.sort_values(
    "f1",
    ascending=False
).reset_index(drop=True)

display(
    rf_table.style.format({
        "accuracy": "{:.6f}",
        "accuracy_std": "{:.4f}",
        "precision": "{:.6f}",
        "precision_std": "{:.4f}",
        "recall": "{:.6f}",
        "recall_std": "{:.4f}",
        "f1": "{:.6f}",
        "f1_std": "{:.4f}",
        "roc_auc": "{:.6f}",
        "roc_auc_std": "{:.4f}",
    })
)

,n_estimators,max_depth,min_samples_split,accuracy,accuracy_std,precision,precision_std,recall,recall_std,f1,f1_std,roc_auc,roc_auc_std
0,100,10,2,0.645264,0.0019,0.243501,0.0015,0.720113,0.0091,0.363933,0.0027,0.733796,0.0031
1,300,10,2,0.643524,0.0021,0.243015,0.0020,0.723124,0.0071,0.363775,0.0030,0.733924,0.0029
2,100,10,5,0.644605,0.0025,0.243214,0.0024,0.720615,0.0054,0.363682,0.0033,0.733744,0.0028
3,300,10,5,0.643355,0.0015,0.242920,0.0020,0.723160,0.0070,0.363676,0.0031,0.733877,0.0029
4,300,5,2,0.621601,0.0036,0.232151,0.0023,0.730114,0.0044,0.352284,0.0031,0.720262,0.0035
5,300,5,5,0.621399,0.0037,0.232025,0.0023,0.729989,0.0040,0.352124,0.0030,0.720296,0.0036
6,100,5,5,0.622164,0.0033,0.231853,0.0021,0.726655,0.0044,0.351538,0.0028,0.720419,0.0037
7,100,5,2,0.622116,0.0032,0.231827,0.0019,0.726655,0.0048,0.351507,0.0026,0.720359,0.0037
